# 경사 하강법 ( Gradient Descent )

- 반복적 접근과 수렴성 : 경사하강법은 반복적으로 기울기를 계산하고 매개변수를 업데이트, 이 과정에서 알고리즘의 수렴성은 중요한 요소로, 적절한 조건하에서 알고리즘은 로컬 미니멈 또는 글로벌 미니멈으로 수렴 
- 학습률(Learning Rate): 학습률은 이동의 크기를 결정 
- 글로벌 미니멈 추구 

## 경사하강법의 종류 

### 배치 경사하강법(Batch Gradient Descent)

- 배치 경사하강법은 매 반복에서 전체 데이터 세트를 사용하여 그래디언트를 계산
- 이 방법은 정확한 그래디언트 방향을 제공, 매우 큰 데이터 세트에 대해서는 계산 비용이 매우 높음

### 미니 배치 경사하강법(Mini-batch Gradient Descent)

- 미니 배치 경사하강법은 전체 데이터 세트들을 작은 배치로 나누고, 각 반복에서 하나의 배치를 사용하여 그래디언트를 계산
- 계산 효율성과 수렴 속도 사이에 좋은 균형을 제공 

### 확률적 경사하강법(Stochastic Gradient Descent)

- 매 반복에서 단 하나의 샘플을 무작위로 선택하여 그래디언트를 계산, 빠른 수렴 속도를 제공하지만, 경로가 불안정 할 수 있으며, 
최소값에 도달하더라도 계속해서 약간씩 변동할 수 있다. 


## SGD


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim # SGD, Adam, RMSprop 등 다양한 최적화 알고리즘 포함

# nn.Linear(in_features, out_features)
# - in_features=1: 입력 특성이 1개인 데이터(x 1개)를 받음
# - out_features=1: 출력값이 1개인 예측(예: y 1개)을 만듦
# 내부적으로 y = wx + b 형태(가중치 w, 편향 b)를 학습하는 가장 기본적인 선형 모델
model = nn.Linear(1, 1)

optimizer = optim.SGD(model.parameters(), lr=0.01) # Stochastic Gradient Descent, SGD
# model.parameters(): 모델의 매개변수(가중치와 편향)를 반환
# lr(learning rate): 한 번 업데이트할 때 이동하는 크기(학습률)

In [3]:
# model 클래스와 내부 정보를 한 번에 확인하는 예제
# 위 셀에서 생성한 model = nn.Linear(1, 1)을 그대로 사용

print("1) 모델 구조")
print(model)

print("\n2) 모델 클래스")
print(type(model))

print("\n3) 파라미터 이름과 shape")
for name, param in model.named_parameters():
    print(f"- {name}: shape={tuple(param.shape)}, value={param.data}")

print("\n4) state_dict 확인")
print(model.state_dict())

# nn.Linear(1, 1)의 현재 수식을 y = wx + b 형태로 출력
w = model.weight.item()
b = model.bias.item()
print("\n5) 현재 선형식")
print(f"y = {w:.6f} * x + {b:.6f}")

1) 모델 구조
Linear(in_features=1, out_features=1, bias=True)

2) 모델 클래스
<class 'torch.nn.modules.linear.Linear'>

3) 파라미터 이름과 shape
- weight: shape=(1, 1), value=tensor([[0.5680]])
- bias: shape=(1,), value=tensor([-0.9249])

4) state_dict 확인
OrderedDict({'weight': tensor([[0.5680]]), 'bias': tensor([-0.9249])})

5) 현재 선형식
y = 0.567972 * x + -0.924855


- 각 학습 스텝(epoch)에서 모델의 손실(loss)를 계산하고, .backward() 메소드를 호출하여 매개변수의 그래디언트를 계산
- 그런 다음 optimize.step()을 호출하여 계산된 그래디언트를 사용하여 매개변수를 업데이트 

```python
for epoch in range(num_epochs):
    # 모델의 예측값 계산
    prediction = model(x_train)

    # 손실(loss) 계산
    loss = criterion(prediction, y_train)

    # 기울기(gradient) 초기화
    optimizer.zero_grad()

    # 역전파(backpropagation) 수행
    loss.backward()

    # 가중치 업데이트
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')
```

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# 1) 학습 데이터 정의
# x: 입력(feature), y: 정답(label)
# 여기서는 y = 2x 관계를 갖는 매우 단순한 선형 데이터 사용
x = torch.tensor([[1.0], [2.0], [3.0]])
y = torch.tensor([[2.0], [4.0], [6.0]])

# 2) 실험 A: 학습률(lr)=0.01
# 목적: 상대적으로 큰 학습률에서 손실이 얼마나 빠르게 줄어드는지 확인
print("[1] 학습률이 0.01인 훈련")
model_01 = nn.Linear(1, 1)  # y = wx + b 형태의 선형 모델
optimizer_01 = optim.SGD(model_01.parameters(), lr=0.01)

# 요청한 체크포인트: 1, 50, 99 epoch에서 상태 출력
check_epochs = {1, 50, 99}

for epoch in range(100):
    # (a) 순전파: 현재 파라미터로 예측값 계산
    pred = model_01(x)

    # (b) 손실 계산: 예측값과 정답의 차이를 MSE로 측정
    loss = F.mse_loss(pred, y)

    # (c) 이전 step에서 누적된 gradient 초기화
    optimizer_01.zero_grad()

    # (d) 역전파: loss를 파라미터(w, b)로 미분하여 gradient 계산
    loss.backward()

    # (e) 파라미터 업데이트: w, b <- w, b - lr * gradient
    optimizer_01.step()

    # 학습 진행 상황 출력
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

    # 요청한 epoch에서 pred/loss/optimizer_01 상태를 자세히 출력
    current_epoch = epoch + 1
    if current_epoch in check_epochs:
        w = model_01.weight.item()
        b = model_01.bias.item()
        lr = optimizer_01.param_groups[0]["lr"]

        # gradient는 backward 직후 값이므로, 해당 시점 값 확인 가능
        w_grad = model_01.weight.grad.item()
        b_grad = model_01.bias.grad.item()

        print(f"\n[Checkpoint Epoch {current_epoch}]")
        print(f"pred: {pred.detach().squeeze().tolist()}")
        print(f"loss: {loss.item():.6f}")
        print(f"optimizer_01 lr: {lr}")
        print(f"weight/bias: w={w:.6f}, b={b:.6f}")
        print(f"grad(weight)/grad(bias): dw={w_grad:.6f}, db={b_grad:.6f}")

print("\n")  # 두 실험의 출력을 구분하기 위한 줄바꿈

# 3) 실험 B: 학습률(lr)=0.001
# 목적: 더 작은 학습률에서 더 안정적이지만 느린 수렴을 확인
print("[2] 학습률이 0.001인 훈련")
model_001 = nn.Linear(1, 1)
optimizer_001 = optim.SGD(model_001.parameters(), lr=0.001)

for epoch in range(100):
    pred = model_001(x)
    loss = F.mse_loss(pred, y)
    optimizer_001.zero_grad()
    loss.backward()
    optimizer_001.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

[1] 학습률이 0.01인 훈련

[Checkpoint Epoch 1]
pred: [0.15899384021759033, 0.6082512140274048, 1.0575085878372192]
loss: 13.107162
optimizer_01 lr: 0.01
weight/bias: w=0.605604, b=-0.222429
grad(weight)/grad(bias): dw=-15.634653, db=-6.783498
Epoch 20, Loss: 0.1642826795578003
Epoch 40, Loss: 0.01418210193514824

[Checkpoint Epoch 50]
pred: [2.157085657119751, 4.024280071258545, 5.891474723815918]
loss: 0.012348
optimizer_01 lr: 0.01
weight/bias: w=1.867994, b=0.289405
grad(weight)/grad(bias): dw=-0.079953, db=0.048560
Epoch 60, Loss: 0.011656197719275951
Epoch 80, Loss: 0.010575275868177414

[Checkpoint Epoch 99]
pred: [2.1452536582946777, 4.031141757965088, 5.91702938079834]
loss: 0.009651
optimizer_01 lr: 0.01
weight/bias: w=1.886164, b=0.258743
grad(weight)/grad(bias): dw=-0.027583, db=0.062283
Epoch 100, Loss: 0.0096044996753335


[2] 학습률이 0.001인 훈련
Epoch 20, Loss: 25.016082763671875
Epoch 40, Loss: 16.032949447631836
Epoch 60, Loss: 10.28302001953125
Epoch 80, Loss: 6.602532863616943
Ep

## 선형회귀 + SGD 학습 정형화 절차

아래 절차를 그대로 따르면 대부분의 기본 딥러닝 학습 코드를 같은 구조로 작성할 수 있습니다.

### 1. 문제/데이터 정의
- 입력 텐서 `x`, 정답 텐서 `y`를 준비한다.
- 텐서 shape를 먼저 확인한다. (예: `(N, 1)`)

### 2. 모델 정의
- 문제에 맞는 모델을 만든다.
- 이 예제는 `nn.Linear(1, 1)`로 `y = wx + b`를 학습한다.

### 3. 손실 함수와 옵티마이저 정의
- 손실 함수: 예측이 정답과 얼마나 다른지 수치화
- 옵티마이저: gradient를 사용해 파라미터를 어떻게 업데이트할지 결정
- 예: `F.mse_loss`, `optim.SGD(model.parameters(), lr=...)`

### 4. 학습 루프 반복
각 epoch마다 아래 순서를 반드시 지킨다.

1. `pred = model(x)` : 순전파(예측)
2. `loss = criterion(pred, y)` : 손실 계산
3. `optimizer.zero_grad()` : gradient 초기화
4. `loss.backward()` : 역전파로 gradient 계산
5. `optimizer.step()` : 파라미터 업데이트

### 5. 로그/평가
- 일정 주기마다 loss를 출력해 수렴 여부를 확인한다.
- 필요하면 학습률별 실험(예: `0.01`, `0.001`)로 속도/안정성을 비교한다.

### 6. 실무 체크포인트
- `optimizer`는 반드시 해당 모델의 `parameters()`를 받게 연결한다.
- 학습률이 너무 크면 발산, 너무 작으면 수렴이 매우 느릴 수 있다.
- 재현성을 위해 `torch.manual_seed(...)`를 설정하는 습관이 좋다.